## Monitoring Traffic Data to Reduce Emissions

Sample Data from 23,204 vehicles passing 1 Intersection across multiple different routes within a timeline of 12 Hours (8 AM - 8 PM).

Data Sourced from https://traffic-signal-control.github.io/

---
**Data Dictionary**
* length: length of the vehicle
* width: width of the vehicle
* maxPosAcc: maximum acceleration (in m/s)
* maxNegAcc: maximum deceleration (in m/s)
* usualPosAcc: usual acceleration (in m/s)
* usualNegAcc: usual deceleration (in m/s)
* minGap: minimum acceptable gap with leading vehicle (in meter)
* maxSpeed: maximum cruising speed (in m/s)
* headwayTime: desired headway time (in seconds) with leading vehicle, keep current speed * headwayTime gap.

**References**
* Data Dictionary : https://cityflow.readthedocs.io/en/latest/flow.html
* http://cires1.colorado.edu/jimenez/Papers/Jimenez_VSP_9thCRC_99_final.pdf
* https://www.mdpi.com/2073-4433/11/7/765/pdf
* https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6801648/pdf/ijerph-16-03647.pdf
* https://onlinelibrary.wiley.com/doi/pdf/10.1002/atr.1215
* https://ecoscore.be/en/info/ecoscore/co2
* https://www.mdpi.com/2071-1050/10/11/3891/s1

---
#### Import the Python Libraries

In [78]:
import pandas as pd
import numpy as np
import math

#### Import Sample Data

In [79]:
df = pd.read_csv("traffic_data_12_hours.csv")

In [80]:
### Random Idling Time between 0 to 30 Seconds

In [81]:
np.random.seed(100)
df['idling'] = np.random.randint(10, 30, df.shape[0])

In [82]:
df['24hour'] = df.apply(lambda x: math.ceil(x["startTime"]/3600+8),axis=1)

---
#### Calculating Vehicle Specific Power

It is an estimate of the power demand on the engine during driving. It is measured in kW/Metric Ton.

It is calculated using the second-by-second speed values in a driving schedule, along with information about the type of vehicle being operated.

**vsp = speed * (a * accel + (g * slope) + b) + (c * speed^3)**

PVSP is the Vehicle Specific Power; speed is the vehicle instantaneous speed; accel is the vehicle acceleration

By default: a = 1.1, b = 0.132, c = 0.000302 and g = 0.132 (as of Jimenez-Palacios, 1999).

In [83]:
def vsp(speed,accel):
    return math.floor(speed * ((1.1 * accel) + (.132) + 0.132) + (0.000302 * (speed**3)))


In [84]:
df["vsp"] = df.apply(lambda x: vsp(x['maxSpeed'],x['usualPosAcc']),axis=1)

#### Creating DataFrames for the VSP Modes and Fuel Consumption averages

In [85]:
vsp_modes = pd.DataFrame({'vsp_mode':[1,2,3,4,5,6,7,8,9,10,11,12,13,14],
                          'wkg_low':[-5,-2,0,1,4,7,10,13,16,19,23,28,33,39],
                         'wkg_high':[-3,-1,0,3,6,9,12,15,18,22,27,32,38,40]})

In [86]:
fuel_consumption = pd.DataFrame({'vsp_mode':[1,2,3,4,5,6,7,8,9,10,11],
                                'petrol(gm/sec)':[.33,.5,.71,1,1.2,1.35,1.48,1.65,1.8,1.85,1.9]})

The Instantaneous Fuel Consumption display shows the value of instantaneous fuel consumption (the kms per liter your car is getting right at that very moment when your car is moving). The display shows estimated values.

In [87]:
df1 = pd.merge(df,vsp_modes,how="left",left_on="vsp",right_on="wkg_high")
final = pd.merge(df1,fuel_consumption,how="left",on="vsp_mode")

In [88]:
final.head()

,interval,startTime,endTime,route1,route2,length,width,maxPosAcc,maxNegAcc,usualPosAcc,...,minGap,maxSpeed,headwayTime,idling,24hour,vsp,vsp_mode,wkg_low,wkg_high,petrol(gm/sec)
0,5,1,1,road_1_2_3,road_1_1_3,5,2,2,4.5,2,...,2.5,11.11,2,18,9,27,11,23,27,1.9
1,5,16,16,road_2_1_2,road_1_1_2,5,2,2,4.5,2,...,2.5,11.11,2,13,9,27,11,23,27,1.9
2,5,22,22,road_2_1_2,road_1_1_2,5,2,2,4.5,2,...,2.5,11.11,2,17,9,27,11,23,27,1.9
3,5,24,24,road_2_1_2,road_1_1_2,5,2,2,4.5,2,...,2.5,11.11,2,25,9,27,11,23,27,1.9
4,5,27,27,road_0_1_0,road_1_1_0,5,2,2,4.5,2,...,2.5,11.11,2,26,9,27,11,23,27,1.9


#### Function to calculate emissions
For a light vehicle, approximate Fuel consumption during idling is 1.89 ltrs per hour. This is calculated as 3.93 grams of carbon emission per second.

1 liter of petrol weighs 750 grammes. Petrol consists for 87% of carbon, or 652 grammes of carbon per liter of petrol. In order to combust this carbon to CO2, 1740 grammes of oxygen is needed. The sum is then 652 + 1740 = 2392 grammes of CO2/liter of petrol.

In [89]:
def co2_emissions(petrol,idle):
    co2 = (petrol + (3.93*idle)) *.87 * 2.66
    return co2

##### Calculate Column for CO2 Emissions

In [90]:
final["CO2/ltr"] = final.apply(lambda x: co2_emissions(x["petrol(gm/sec)"],x["idling"]),axis=1)

---
##### Estimated Total Emissions during a 60 Minute Period

In [91]:
print(f'{round(final["CO2/ltr"].sum()/1000,2)}kg CO2')

4215.82kg CO2


---
### Splitting Data

#### Testing and Training Data

Choose the the Top 3 Ranked Variables in the Independent Variable Split

In [92]:
X = final[['24hour','idling','petrol(gm/sec)']]

y = final[['CO2/ltr']]
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.3,random_state=100)

---
## Build the Model
#### K-Nearest Neighbor

In [93]:
from sklearn.neighbors import KNeighborsRegressor

model = KNeighborsRegressor(n_neighbors=math.floor(math.sqrt(len(X_train))))

model.fit(X_train,y_train)

KNeighborsRegressor(algorithm='auto', leaf_size=30, metric='minkowski',
                    metric_params=None, n_jobs=None, n_neighbors=127, p=2,
                    weights='uniform')

#### Apply the Model

In [94]:
y_predict = model.predict(X_test)

##### Check Model Accuracy on Test Data

In [95]:
f'{round(model.score(X_test,y_test) * 100,2)}%'

'99.79%'

##### Check Model Accuracy on Training Data

In [96]:
f'{round(model.score(X_train,y_train) * 100,2)}%'

'99.8%'

##### Create a New DataFrame with Actual and Predicted Data

In [97]:
pred = y_test.copy()
pred.insert(1,"Prediction",y_predict)

In [98]:
pred.rename(columns={"Predicted":"Actual"},inplace=True)

In [99]:
pred.head()

,CO2/ltr,Prediction
9624,95.345040,97.135356
6668,186.293100,186.722776
18113,249.956742,246.734173
13743,95.345040,97.708257
15721,231.767130,234.273573


#### Apply a Prediction
Use the selected variables to apply the prediction for CO2 Emissions

In [110]:
hour = 9
idling = 30
petrol_gm_sec = 1.8

model.predict([[hour,idling,petrol_gm_sec]])[0][0]

267.93151606299216